# 16 — Topic Modeling with NMF

**Learning objective.** Discover latent themes in an unlabeled corpus and interpret topic-word/document-topic matrices.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**unlabeled corpus → matrix factorization → topic mixtures → exploratory themes**

Focus on the transformation of information from left to right. Ask what representation changes before asking which library call implements it.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Change number of **topics** | factorization granularity changes | themes split or merge |
| Change representation/stopwords | word weights change | topic descriptors can change dramatically |
| Change random seed/initialization | local solution can change | topic stability becomes a validation question |

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. If you ask for too many topics on a tiny corpus, what do you expect?
2. Why is a model's 'topic 2' not automatically a real business category?

### When to use
Useful for exploratory structure discovery when labels do not exist.

### When not to use / caution
Do not treat topics as ground truth classes without human validation.

### Debugging lens
Check coherence, stability and representative documents—not only top words.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
docs=[
 'bank card payment refund fee','credit card charge transaction bank','refund payment merchant card',
 'neural network deep learning model','transformer attention language model','training neural model gpu',
 'flight hotel travel booking','hotel room travel airport','booking flight airport ticket']
vec=TfidfVectorizer(stop_words='english')
X=vec.fit_transform(docs)
nmf=NMF(n_components=3,random_state=42,init='nndsvda',max_iter=500)
W=nmf.fit_transform(X); H=nmf.components_; terms=np.array(vec.get_feature_names_out())
for k,row in enumerate(H):
    print('topic',k,':',', '.join(terms[row.argsort()[-5:][::-1]]))
print('doc topics:',W.argmax(axis=1).tolist())

topic 0 : travel, flight, hotel, booking, airport
topic 1 : card, refund, payment, bank, merchant
topic 2 : model, neural, gpu, training, network
doc topics: [1, 1, 1, 2, 2, 2, 0, 0, 0]


Topic numbers are not labels until a human interprets them. Stability across seeds/time windows matters more than a visually appealing single run.

---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Interpret topic-word and document-topic matrices
- Treat topic labels as human interpretations, not model ground truth